### SVM(Support Vector Machine)
- 선을 구성하는 매개변수를 조정해서 요소들을 구분하고 선을 찾고, 이를 기반으로 패턴을 인식
- 패턴들과의 거리(마진)를 최대로 만드는 것이 가장 좋은 결과를 얻음.

---
#### SVM을 활용한 BMI 측정 에측

#### BMI Data생성

In [28]:
# 데이터를 획득하기 위해 무작위로 2만명 데이터 생성
# 키(cm), 몸무게(kg), Label[저체중(thin), 정상체중(normal), 비만(fat)]의 csv file
# bmi가 18.5 미만 : thin
# bmi가 25미만 : normal
# 아니면 fat
# BMI = 체중(kg) / (신장(m) * 신장(m)). 

import random

# BMI를 계산해서 레이블(결과)를 리턴하는 함수
def calc_bmi(h, w):
  bmi = w / (h / 100) ** 2
  # if bmi < 18.5:
  #   return 'thin'
  # elif bmi < 25:
  #   return 'normal'
  # else:
  #   return 'fat'
  if bmi < 18.5: return "thin"
  if bmi < 25: return "normal"
  return "fat"


# 출력 파일
fp = open("../Data/bmi.csv", "w", encoding='utf-8')
fp.write("height,weight,lable\n")


# 무작위로 데이터 생성
cnt = {'thin':0, 'normal':0, 'fat':0}

for i in range(20000):
  h = random.randint(120, 200)
  w = random.randint(35,80)
  label = calc_bmi(h,w)
  cnt[label]+=1
  fp.write(f"{h},{w},{label}\n")
fp.close()
print("OK,",cnt)

OK, {'thin': 6394, 'normal': 5917, 'fat': 7689}


In [29]:
5917 / 7689

0.7695409025881129

### SVM으로 확인

In [30]:
import pandas as pd

In [31]:
tbl = pd.read_csv("../Data/bmi.csv")
tbl.head()

,height,weight,lable
0,153,52,normal
1,153,75,fat
2,137,77,fat
3,139,53,fat
4,183,47,thin


In [36]:
# 정규화
label = tbl['lable']
w = tbl['weight'] / tbl['weight'].max()
h = tbl['height'] / tbl['height'].max()

wh = pd.concat(
  [w,h],
  axis='columns'
)
wh.head()

,weight,height
0,0.6500,0.765
1,0.9375,0.765
2,0.9625,0.685
3,0.6625,0.695
4,0.5875,0.915


In [37]:
# Training과 Test Data 분리하기
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import collections # 빈도수 세기

In [38]:
train_data, test_data, train_target, test_target = \
  train_test_split(
    wh,
    label,
    random_state=42,
    stratify=label,
    test_size=0.2
  )

In [40]:
# Training

from sklearn.svm import SVC
from sklearn import metrics
clf = SVC()

clf.fit(train_data, train_target)
print("Train score:", clf.score(train_data, train_target))
print("Test score:", clf.score(test_data, test_target))
clf.score

Train score: 0.9951875
Test score: 0.9945


<bound method ClassifierMixin.score of SVC()>

In [41]:
# 예측값을 구하기
pred = clf.predict(test_data)
pred

array(['fat', 'normal', 'normal', ..., 'fat', 'normal', 'normal'],
      dtype=object)

In [42]:
# 예측값과 실제 test target과 동일한 데이터의 평균이 정확도와 일치하는지 확인
(pred == test_target).mean()

0.9945

In [43]:
clf_report = metrics.classification_report(test_target, pred)
print(clf_report)

              precision    recall  f1-score   support

         fat       1.00      0.99      1.00      1538
      normal       0.99      1.00      0.99      1183
        thin       1.00      0.99      1.00      1279

    accuracy                           0.99      4000
   macro avg       0.99      0.99      0.99      4000
weighted avg       0.99      0.99      0.99      4000

